# Notebook 4: Agent Evaluation

Evaluate the agent from Module 6 across three dimensions: **tool routing** (did it call the right tools?), **argument correctness** (did it pass the right arguments?), and **end-to-end answer quality** (is the final answer correct?).

## Setup

In [1]:
import json
import sys
import math
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

# The agent tools live in 6_Agents/tools — reuse them directly.
sys.path.insert(0, str(Path("../../6_Agents").resolve()))
from tools import TOOL_SCHEMAS, TOOL_FUNCTIONS

load_dotenv()
client          = OpenAI()
MODEL           = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
DATA_PATH       = Path("../data/AI_Agent_Insure.md")
GOLDEN_PATH     = Path("../data/golden_dataset.json")
TOP_K           = 8

## Part 1: Rebuild the agent from Module 6

Same agent loop as `6_Agents/notebooks/3_Agent_App.ipynb` — we're evaluating the same system.

In [2]:
def load_and_chunk(path: Path) -> list[str]:
    doc    = path.read_text(encoding="utf-8").strip()
    chunks = [s.strip() for s in doc.replace("\n", " ").split(".") if s.strip()]
    return [c + "." for c in chunks]


chunks     = load_and_chunk(DATA_PATH)
resp       = client.embeddings.create(model=EMBEDDING_MODEL, input=chunks)
embeddings = [r.embedding for r in resp.data]

chroma = chromadb.EphemeralClient()
coll   = chroma.create_collection("evals_agent")
coll.add(
    ids=[str(i) for i in range(len(chunks))],
    embeddings=embeddings,
    documents=chunks
)


def search_docs(query: str) -> str:
    q_emb   = client.embeddings.create(model=EMBEDDING_MODEL, input=[query])
    q_vec   = q_emb.data[0].embedding
    results = coll.query(query_embeddings=[q_vec], n_results=TOP_K)
    docs    = results["documents"][0]
    return "\n---\n".join(docs) if docs else "No relevant content found."


SEARCH_DOCS_SCHEMA = {
    "type": "function",
    "function": {
        "name": "search_docs",
        "description": (
            "Search the AI Agent Insure company document for detailed information. "
            "Use this for questions about company background, mission, operational model, "
            "claims process, or anything not covered by the other structured tools."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query."}
            },
            "required": ["query"]
        }
    }
}

ALL_SCHEMAS   = TOOL_SCHEMAS + [SEARCH_DOCS_SCHEMA]
ALL_FUNCTIONS = {**TOOL_FUNCTIONS, "search_docs": search_docs}

SYSTEM_PROMPT = (
    "You are a helpful assistant for AI Agent Insure, a specialty insurer for "
    "AI systems, autonomous agents, and ML infrastructure. "
    "Use search_docs to look up detailed information from the company document. "
    "Use the other tools for structured lookups (product details, pricing, eligibility). "
    "Call as many tools as you need — do not guess when a tool can give you the answer."
)

print(f"Agent ready with {len(ALL_SCHEMAS)} tools: {[s['function']['name'] for s in ALL_SCHEMAS]}")

Agent ready with 4 tools: ['get_product_info', 'get_pricing_estimate', 'check_eligibility', 'search_docs']


In [3]:
def run_agent(user_question: str, max_iterations: int = 10) -> tuple[str, list[dict]]:
    """
    Run the agent and return (final_answer, trace).
    trace is a list of {"tool": str, "args": dict, "result": str}.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_question}
    ]
    trace = []

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL, messages=messages,
            tools=ALL_SCHEMAS, tool_choice="auto", temperature=0
        )
        choice = response.choices[0]
        messages.append(choice.message)

        if choice.finish_reason == "stop":
            return choice.message.content, trace

        if choice.finish_reason == "tool_calls":
            for tc in choice.message.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments)
                result  = ALL_FUNCTIONS[fn_name](**fn_args)
                trace.append({"tool": fn_name, "args": fn_args, "result": result})
                messages.append({
                    "role": "tool", "tool_call_id": tc.id, "content": result
                })

    return f"[Stopped after {max_iterations} iterations]", trace

## Part 2: Tool routing accuracy

For each golden question, compare the tools the agent *actually called* against the tools it *should have called*.

We measure this as a set comparison — order doesn't matter, only which tools were called.

In [4]:
def routing_score(actual_tools: list[str], expected_tools: list[str]) -> float:
    """
    F1-style score over the tool sets.
    1.0 = exact match, 0.0 = no overlap.
    """
    actual   = set(actual_tools)
    expected = set(expected_tools)
    if not expected:
        return 1.0 if not actual else 0.0
    if not actual:
        return 0.0
    precision = len(actual & expected) / len(actual)
    recall    = len(actual & expected) / len(expected)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


golden = json.loads(GOLDEN_PATH.read_text(encoding="utf-8"))

print("Running agent eval — tool routing...")
print()

routing_results = []
for case in golden:
    answer, trace = run_agent(case["question"])
    actual_tools  = [step["tool"] for step in trace]
    score         = routing_score(actual_tools, case["expected_tools"])

    routing_results.append({
        "id":       case["id"],
        "category": case["category"],
        "question": case["question"][:55],
        "expected": case["expected_tools"],
        "actual":   actual_tools,
        "score":    round(score, 2),
        "answer":   answer,
        "expected_answer": case["expected_answer"],
    })

print(f"{'ID':>4}  {'Cat':12}  {'Score':>5}  {'Expected':30}  Actual")
print("-" * 90)
for r in routing_results:
    flag = "  " if r["score"] == 1.0 else " !"
    print(f"{r['id']:>4}  {r['category']:12}  {r['score']:>5}{flag}  {str(r['expected']):30}  {r['actual']}")

mean_routing = sum(r["score"] for r in routing_results) / len(routing_results)
print()
print(f"Mean routing score: {mean_routing:.3f}")

Running agent eval — tool routing...

  ID  Cat           Score  Expected                        Actual
------------------------------------------------------------------------------------------
 q01  product_info    1.0    ['get_product_info']            ['get_product_info']
 q02  pricing         1.0    ['get_pricing_estimate']        ['get_pricing_estimate']
 q03  eligibility     1.0    ['check_eligibility']           ['check_eligibility']
 q04  rag             1.0    ['search_docs']                 ['search_docs']
 q05  rag             1.0    ['search_docs']                 ['search_docs']
 q06  product_info    1.0    ['get_product_info']            ['get_product_info']
 q07  pricing         1.0    ['get_pricing_estimate']        ['get_pricing_estimate']
 q08  eligibility     1.0    ['check_eligibility']           ['check_eligibility']
 q09  multi_tool      1.0    ['check_eligibility', 'get_pricing_estimate']  ['check_eligibility', 'get_pricing_estimate']
 q10  multi_tool      1.0  

## Part 3: End-to-end answer quality

Use semantic similarity (from Notebook 3) to score the agent's final answer against the expected answer. This is the end-to-end signal — it doesn't tell you *where* things went wrong, but it tells you *if* they did.

In [5]:
def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot    = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


def semantic_similarity(text_a: str, text_b: str) -> float:
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text_a, text_b])
    return cosine_similarity(resp.data[0].embedding, resp.data[1].embedding)


print("Scoring end-to-end answer quality...")
print()

e2e_results = []
for r in routing_results:
    # Find the original golden entry to get expected_answer.
    golden_entry = next(e for e in golden if e["id"] == r["id"])
    sim = semantic_similarity(r["answer"], golden_entry["expected_answer"])
    e2e_results.append({**r, "e2e_sim": round(sim, 3)})

print(f"{'ID':>4}  {'Cat':12}  {'Route':>5}  {'E2E':>5}  Question")
print("-" * 80)
for r in e2e_results:
    flag = "  " if r["e2e_sim"] >= 0.75 else " !"
    print(f"{r['id']:>4}  {r['category']:12}  {r['score']:>5}  {r['e2e_sim']:>5}{flag}  {r['question']}")

mean_e2e = sum(r["e2e_sim"] for r in e2e_results) / len(e2e_results)
print()
print(f"Mean routing score : {mean_routing:.3f}")
print(f"Mean E2E similarity: {mean_e2e:.3f}")

Scoring end-to-end answer quality...

  ID  Cat           Route    E2E  Question
--------------------------------------------------------------------------------
 q01  product_info    1.0  0.999    What does Agentic AI Liability Insurance cover?
 q02  pricing         1.0  0.913    How much would Model & Data Security Insurance cost for
 q03  eligibility     1.0  0.871    Can a healthcare company get coverage from AI Agent Ins
 q04  rag             1.0  0.858    How does AI Agent Insure handle claims?
 q05  rag             1.0  0.868    What is AI Agent Insure's underwriting philosophy?
 q06  product_info    1.0  0.875    What does Autonomous Systems & Robotics Coverage includ
 q07  pricing         1.0  0.925    How much does Agentic AI Liability Insurance cost for a
 q08  eligibility     1.0  0.841    Is a synthetic data company eligible for coverage?
 q09  multi_tool      1.0  0.623 !  I'm a robotics startup. Am I eligible and what would co
 q10  multi_tool      1.0  0.755    Explain 

## Part 4: Diagnose a failure

When routing score and E2E score disagree, that's a signal about *where* the failure is.

In [6]:
print("Failure analysis")
print("=" * 60)
print()

for r in e2e_results:
    routing_ok = r["score"] == 1.0
    e2e_ok     = r["e2e_sim"] >= 0.75

    if routing_ok and not e2e_ok:
        diagnosis = "GENERATION FAILURE — right tools, wrong answer"
    elif not routing_ok and e2e_ok:
        diagnosis = "ROUTING MISS — wrong tools, but answer still OK (lucky)"
    elif not routing_ok and not e2e_ok:
        diagnosis = "ROUTING + GENERATION FAILURE"
    else:
        diagnosis = "PASS"

    if diagnosis != "PASS":
        print(f"[{r['id']}] {diagnosis}")
        print(f"  Q        : {r['question']}")
        print(f"  Expected tools: {r['expected']}")
        print(f"  Actual tools  : {r['actual']}")
        print(f"  E2E sim  : {r['e2e_sim']}")
        print()

Failure analysis

[q09] GENERATION FAILURE — right tools, wrong answer
  Q        : I'm a robotics startup. Am I eligible and what would co
  Expected tools: ['check_eligibility', 'get_pricing_estimate']
  Actual tools  : ['check_eligibility', 'get_pricing_estimate']
  E2E sim  : 0.623



## Part 5: The three diagnostic patterns

| Routing | E2E | Diagnosis | Fix |
|---|---|---|---|
| ✓ | ✓ | Pass | — |
| ✓ | ✗ | Generation failure | Improve system prompt or generation step |
| ✗ | ✓ | Routing miss, got lucky | Fix routing — it will fail on harder cases |
| ✗ | ✗ | Routing + generation failure | Fix routing first, then re-evaluate generation |

This is why you measure each layer separately. End-to-end alone can mask routing problems that will surface on harder questions.

## Part 6: Summary — all three eval layers together

In [7]:
passes = sum(1 for r in e2e_results if r["score"] == 1.0 and r["e2e_sim"] >= 0.75)

print("Eval summary")
print("=" * 40)
print(f"Total test cases        : {len(e2e_results)}")
print(f"Routing score (mean F1) : {mean_routing:.3f}")
print(f"E2E similarity (mean)   : {mean_e2e:.3f}")
print(f"Full passes             : {passes} / {len(e2e_results)}")
print()
print("This is your baseline. Run this again after any change to see if things improved.")

Eval summary
Total test cases        : 10
Routing score (mean F1) : 1.000
E2E similarity (mean)   : 0.853
Full passes             : 9 / 10

This is your baseline. Run this again after any change to see if things improved.


## Key concepts

- Agent evaluation has three layers: routing, argument correctness, and end-to-end quality
- Routing and E2E can disagree — measuring both tells you *where* the failure is
- The eval suite from all four notebooks is your regression baseline — run it before and after any change
- **Next:** The eval app packages this into a Streamlit UI you can run on demand